In [1]:
import pandas as pd
import numpy as np

dataset_path = "../griffin_datasets/talk"
griffin_path = "../griffin_datasets/talk-pk"



In [4]:
brand = pd.read_parquet(f"{dataset_path}/pbdm.pqt")
brand["primary_key"] = np.arange(len(brand))
brand

,device_id,phone_brand,device_model,primary_key
0,-8890648629457979026,小米,红米,0
1,1277779817574759137,小米,MI 2,1
2,5137427614288105724,三星,Galaxy S4,2
3,3669464369358936369,SUGAR,时尚手机,3
4,-5019277647504317457,三星,Galaxy Note 2,4
...,...,...,...,...
187240,7979541072208733273,小米,MI 4,187240
187241,-187404680852357705,小米,红米2,187241
187242,-2718274279595622821,小米,MI 3,187242
187243,3098391762071677791,vivo,X1,187243


In [5]:
App_labels = pd.read_parquet(f"{dataset_path}/alabel.pqt")
App_labels["primary_key"] = np.arange(len(App_labels))
App_labels


,app_id,label_id,primary_key
0,7324884708820027918,251,0
1,-4494216993218550286,251,1
2,6058196446775239644,406,2
3,6058196446775239644,407,3
4,8694625920731541625,406,4
...,...,...,...
459938,8899923330228547048,932,459938
459939,8906927134640865376,932,459939
459940,9180397886043162136,932,459940
459941,9204453858293451002,932,459941


In [6]:
App_events = pd.read_parquet(f"{dataset_path}/aevent.pqt")
App_events["primary_key"] = np.arange(len(App_events))
App_events



,event_id,app_id,is_installed,is_active,primary_key
0,2,5927333115845830913,1,1,0
1,2,-5720078949152207372,1,0,1
2,2,-1633887856876571208,1,0,2
3,2,-653184325010919369,1,1,3
4,2,8693964245073640147,1,1,4
...,...,...,...,...,...
32473062,3252948,6607018907660377991,1,1,32473062
32473063,3252948,6602285879264922467,1,1,32473063
32473064,3252948,4348659952760821294,1,1,32473064
32473065,3252948,-995726944612374565,1,1,32473065


In [8]:
# Save the parquet files under griffin_dataset
import shutil
import os
shutil.rmtree(griffin_path, ignore_errors=True)
shutil.copytree(dataset_path, griffin_path)


'../griffin_datasets/talk-pk'

In [9]:
brand.to_parquet(f"{griffin_path}/pbdm.pqt")
App_labels.to_parquet(f"{griffin_path}/alabel.pqt")
App_events.to_parquet(f"{griffin_path}/aevent.pqt")


dataset_name: talkingdata
tables:
  - name: Gender_age
    source: gender_age.pqt
    format: parquet
    columns:
    - name: device_id
      dtype: primary_key
  - name: Brand
    source: pbdm.pqt
    format: parquet
    columns:
    - name: primary_key
      dtype: primary_key
    - name: device_id
      dtype: foreign_key
      link_to: Gender_age.device_id
    - name: phone_brand
      dtype: category
    - name: device_model
      dtype: category
  - name: App_labels
    source: alabel.pqt
    format: parquet
    columns:
    - name: primary_key
      dtype: primary_key
    - name: app_id
      dtype: foreign_key
      link_to: Apps.app_id
    - name: label_id
      dtype: foreign_key
      link_to: Label_categories.label_id
  - name: App_events
    source: aevent.pqt
    format: parquet
    columns:
    - name: primary_key
      dtype: primary_key
    - name: event_id
      dtype: foreign_key
      link_to: Events.event_id
    - name: app_id
      dtype: foreign_key
      link_to: Apps.app_id
    - name: is_active
      dtype: category
  - name: Label_categories
    source: label_cat.pqt
    format: parquet
    columns:
    - name: label_id
      dtype: primary_key
    - name: category
      dtype: text
  - name: Events
    source: event.pqt
    format: parquet
    columns:
    - name: event_id
      dtype: primary_key
    - name: device_id
      dtype: foreign_key
      link_to: Gender_age.device_id
    - name: timestamp
      dtype: datetime
    - name: longitude
      dtype: float
    - name: latitude
      dtype: float
    time_column: timestamp
tasks:
  - name: demo-pred
    source: demo-pred/{split}.pqt
    format: parquet
    columns:
      - name: device_id
        dtype: primary_key
      - name: group
        dtype: category
    time_column: null
    evaluation_metric: logloss
    target_column: group
    target_table: Gender_age
    task_type: classification